# Dasar Machine Learning

## Bagian 1d: Feature Engineering


## Feature Engineering pada Data Perumahan Ames

Data yang dipakai berasal dari Ames Housing Dataset yang disusun Dean De Cock (Truman State University, *Journal of Statistics Education* vol. 19 no. 3, 2011).

Notebook ini membangun model regresi linear untuk memprediksi harga rumah, lalu memperbaiki performanya lewat **feature engineering**, yaitu menurunkan fitur baru dari data yang sudah ada. Pendekatan ini sering menjadi pembeda antara model lemah dan model kuat.

**Isi notebook**

1. EDA sederhana
2. One-hot encoding variabel kategorikal
3. Transformasi log untuk variabel miring
4. Pair plot antar fitur
5. Feature engineering dasar: suku polinomial dan interaksi
6. Feature engineering: kategori dan agregat kategori

## 1. EDA sederhana


In [ ]:
# Imports
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures

warnings.filterwarnings("ignore")
sns.set()


#### Memuat data


In [ ]:
# Unduh data sekali saja, lalu muat ke DataFrame
data_path = Path("Ames_Housing_Data.tsv")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML0232EN-SkillsNetwork/asset/Ames_Housing_Data.tsv"

if not data_path.exists():
    urllib.request.urlretrieve(data_url, data_path)

dataset = pd.read_csv(data_path, sep="\t")
dataset.info()


In [ ]:
# Buang beberapa outlier sesuai anjuran penulis dataset
dataset = dataset.loc[dataset["Gr Liv Area"] <= 4000, :]
print(f"Jumlah baris: {dataset.shape[0]}")
print(f"Jumlah kolom: {dataset.shape[1]}")

original_data = dataset.copy()
dataset.head()


Setelah penyaringan, tersisa 2925 baris. Tahap berikutnya adalah pembersihan dasar: mengubah variabel kategorikal menjadi variabel dummy dan menyeimbangkan distribusi variabel yang miring.

### One-hot encoding

Kolom bertipe teks diubah menjadi sekumpulan kolom biner. Satu kategori dibuang (`drop_first=True`) agar tidak terjadi kolinearitas sempurna antar kolom dummy.


In [ ]:
# Kumpulkan nama kolom kategorikal
categorical_columns = dataset.select_dtypes(include="object").columns.tolist()
dataset[categorical_columns].head().T


In [ ]:
# Contoh hasil encoding pada lima baris pertama
pd.get_dummies(dataset[categorical_columns].head(), drop_first=True)


### Transformasi log pada variabel miring

Kemiringan (skewness) dihitung pada semua kolom numerik. Kolom dengan nilai absolut skewness di atas 0.75 dianggap cukup miring untuk ditransformasi.


In [ ]:
skew_limit = 0.75
numeric_columns = dataset.select_dtypes(include=np.number).columns
skew_values = dataset[numeric_columns].skew()
skewed_columns = skew_values[skew_values.abs() > skew_limit].sort_values(ascending=False)
skewed_columns.to_frame(name="Skew")


In [ ]:
# Bandingkan distribusi satu kolom sebelum dan sesudah log1p
field = "BsmtFin SF 1"
figure, (axis_before, axis_after) = plt.subplots(1, 2, figsize=(10, 5))

dataset[field].hist(ax=axis_before)
dataset[field].apply(np.log1p).hist(ax=axis_after)

axis_before.set(title="before np.log1p", ylabel="frequency", xlabel="value")
axis_after.set(title="after np.log1p", ylabel="frequency", xlabel="value")
figure.suptitle(f'Field "{field}"')
plt.show()


In [ ]:
# Terapkan log1p ke seluruh kolom miring, kecuali target
log_columns = [column for column in skewed_columns.index if column != "SalePrice"]
dataset[log_columns] = dataset[log_columns].apply(np.log1p)

pd.get_dummies(dataset, columns=categorical_columns, drop_first=True).shape


In [ ]:
# Nilai yang hilang pada data asli
dataset.isnull().sum().sort_values()


### Fitur numerik untuk ilustrasi

Sebagian kecil kolom numerik dipilih agar transformasi dasar lebih mudah diamati.


In [ ]:
selected_columns = [
    "Lot Area", "Overall Qual", "Overall Cond", "Year Built",
    "Year Remod/Add", "Gr Liv Area", "Full Bath", "Bedroom AbvGr",
    "Fireplaces", "Garage Cars", "SalePrice",
]
subset_data = original_data.loc[:, selected_columns]
subset_data.describe().T


In [ ]:
subset_data.info()


In [ ]:
# Satu nilai kosong pada Garage Cars diisi nol
subset_data = subset_data.fillna(0)
subset_data.info()


### Pair plot fitur

Pair plot memperlihatkan bentuk distribusi target, hubungan setiap fitur dengan target, serta hubungan antar fitur.


In [ ]:
sns.pairplot(subset_data, plot_kws=dict(alpha=0.1, edgecolor="none"))
plt.show()


**Diskusi eksplorasi data**

1. Distribusi `SalePrice` miring ke kanan, jadi belum simetris.
2. Hubungan `Overall Qual` dan `Gr Liv Area` dengan harga melengkung ke atas, bukan garis lurus. Regresi linear masih bisa dipakai bila fitur tersebut ditambah suku kuadrat.
3. Beberapa pasangan fitur saling berkorelasi kuat, misalnya `Garage Cars` dan `Gr Liv Area`. Korelasi antar prediktor dapat membuat koefisien model sulit ditafsirkan.


Target dan fitur dipisahkan ke variabel masing-masing.


In [ ]:
# Pisahkan fitur dan target
features = subset_data.drop(columns=["SalePrice"])
target = subset_data["SalePrice"]

features.info()


Model dievaluasi dengan pembagian data latih dan validasi. Karena pembagian yang sama akan dipakai berulang saat fitur bertambah, langkah tersebut dibungkus dalam satu fungsi agar konsisten.


In [ ]:
def evaluate_features(feature_frame, target_series, test_size=0.2, random_state=42):
    """Latih regresi linear pada satu set fitur dan kembalikan skor R2."""
    features_train, features_valid, target_train, target_valid = train_test_split(
        feature_frame, target_series, test_size=test_size, random_state=random_state
    )
    model = LinearRegression().fit(features_train, target_train)
    return pd.Series(
        {
            "train_r2": model.score(features_train, target_train),
            "valid_r2": model.score(features_valid, target_valid),
        }
    )

baseline_score = evaluate_features(features, target)
baseline_score


### Feature engineering dasar: suku polinomial dan interaksi

Pada pair plot, `Overall Qual` dan `Gr Liv Area` memperlihatkan hubungan melengkung terhadap harga. Suku kuadrat dapat menangkap pola tersebut tanpa meninggalkan kerangka regresi linear.

#### Fitur polinomial


In [ ]:
features_poly = features.copy()
features_poly["OQ2"] = features_poly["Overall Qual"] ** 2
features_poly["GLA2"] = features_poly["Gr Liv Area"] ** 2

evaluate_features(features_poly, target)


Setiap fitur juga bisa saling memengaruhi. Kenaikan `Overall Qual` mungkin lebih bernilai pada rumah yang lebih baru, sehingga hasil kali `Overall Qual` dengan `Year Built` dapat menangkap efek tersebut. Bentuk interaksi lain adalah perbandingan antar fitur, misalnya kualitas per satuan luas lahan.

#### Interaksi antar fitur


In [ ]:
features_interact = features_poly.copy()
features_interact["OQ_x_YB"] = features_interact["Overall Qual"] * features_interact["Year Built"]
features_interact["OQ_/_LA"] = features_interact["Overall Qual"] / features_interact["Lot Area"]

evaluate_features(features_interact, target)


**Latihan interaksi**: interaksi lain apa yang menurutmu berguna, dan mengapa?


### Kategori dan fitur turunan agregat kategori

Variabel kategorikal masuk ke regresi linear lewat kolom dummy berisi 0 dan 1. Sebelum itu, sebaran kategori diperiksa agar tidak ada kategori yang terlalu jarang.


In [ ]:
dataset["House Style"].value_counts()


In [ ]:
pd.get_dummies(dataset["House Style"], drop_first=True).head()


Kolom `Neighborhood` punya 28 kategori. Kategori dengan delapan pengamatan atau kurang digabung menjadi `Other` agar jumlahnya tidak terlalu kecil.


In [ ]:
neighborhood_counts = dataset["Neighborhood"].value_counts()
rare_neighborhoods = list(neighborhood_counts[neighborhood_counts <= 8].index)
rare_neighborhoods


In [ ]:
features_neighborhood = features_interact.copy()
features_neighborhood["Neighborhood"] = dataset["Neighborhood"].replace(rare_neighborhoods, "Other")
features_neighborhood = pd.get_dummies(features_neighborhood, columns=["Neighborhood"], drop_first=True)

evaluate_features(features_neighborhood, target)


#### Fitur berbasis deviasi kategori

Fitur berikut mengukur posisi sebuah nilai terhadap rata-rata kategorinya. Contohnya, seberapa bagus sebuah rumah dibanding rumah lain di lingkungan atau tipe yang sama. Nilai ini dihitung sebagai selisih terhadap rata-rata kategori, lalu dibagi simpangan baku kategori tersebut.


In [ ]:
def add_deviation_feature(frame, feature, category):
    """Tambahkan z-score fitur di dalam kategorinya sebagai kolom baru."""
    grouped = frame.groupby(category)[feature]
    deviation = (frame[feature] - grouped.transform("mean")) / grouped.transform("std")
    frame[f"{feature}_Dev_{category}"] = deviation.fillna(0)
    return frame


In [ ]:
features_deviation = features_interact.copy()
features_deviation["House Style"] = dataset["House Style"]
features_deviation["Neighborhood"] = dataset["Neighborhood"].replace(rare_neighborhoods, "Other")
add_deviation_feature(features_deviation, "Year Built", "House Style")
add_deviation_feature(features_deviation, "Overall Qual", "Neighborhood")
features_deviation = pd.get_dummies(
    features_deviation, columns=["House Style", "Neighborhood"], drop_first=True
)

evaluate_features(features_deviation, target)


## Fitur polinomial di scikit-learn

`PolynomialFeatures` membuat banyak suku berorde tinggi sekaligus. Dengan `degree=2`, hasilnya mencakup intersep, suku linear, suku kuadrat, dan suku silang.


In [ ]:
poly_features = ["Lot Area", "Overall Qual"]
poly = PolynomialFeatures(degree=2)
poly.fit(dataset[poly_features])

poly.get_feature_names_out(input_features=poly_features)


In [ ]:
poly_array = poly.transform(dataset[poly_features])
pd.DataFrame(poly_array, columns=poly.get_feature_names_out(input_features=poly_features))


## Ringkasan

Sebagian besar peningkatan performa model biasanya datang dari sekumpulan kecil fitur yang intuitif dan kuat, seperti transformasi polinomial, interaksi, dan representasi kategorikal. Feature engineering lanjutan masih dapat menambah akurasi, tetapi manfaatnya cenderung mengecil seiring bertambahnya fitur.
